# DNN vs CNN — Image Classification with MNIST

## Goal

Use the **same image dataset and the same ML pipeline** with two different neural-network architectures:

- **Fully Connected DNN**: `Flatten → Dense → Dense`
- **CNN**: `Conv2D → MaxPooling → Flatten → Dense`

The purpose is not to build the most accurate model. The purpose is to **see why CNNs are better suited to image data**.

### What we will compare

1. Image representation
2. Model architecture
3. Number of trainable parameters
4. Training and test performance
5. Predictions on the same images
6. CNN feature maps — what the convolution layer learns

> **Key idea:** The overall ML pipeline stays the same. The major difference is the **model architecture** and how it processes the image.


## 1. Imports and reproducibility

We use TensorFlow/Keras for both models and Matplotlib/NumPy for inspection and visualization.

A fixed seed makes the experiment more reproducible, although exact training results can still vary by hardware/runtime.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)


## 2. Load the MNIST dataset

MNIST contains handwritten digits from **0 to 9**.

Each image is:

- 28 × 28 pixels
- grayscale
- one channel

We deliberately use the **same data** for both the DNN and CNN.


In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


## 3. Inspect one image

Before building a model, look at the actual data.

The image is a 28 × 28 matrix. Each value represents pixel intensity.


In [ ]:
index = 0

plt.figure(figsize=(4, 4))
plt.imshow(X_train[index], cmap="gray")
plt.title(f"Digit: {y_train[index]}")
plt.axis("off")
plt.show()

print("Image shape:", X_train[index].shape)
print("Pixel range:", X_train[index].min(), "to", X_train[index].max())


## 4. Preprocessing — same pipeline for both models

Normalize pixel values from **0–255** to **0–1**.

The only shape difference comes from the CNN needing an explicit channel dimension:

- DNN input: `(28, 28)`
- CNN input: `(28, 28, 1)`

The underlying images are still exactly the same.


In [ ]:
# Normalize pixel values to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# CNN expects: height × width × channels
X_train_cnn = X_train[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]

print("DNN input shape:", X_train.shape)
print("CNN input shape:", X_train_cnn.shape)


## 5. What does the DNN see?

A fully connected DNN cannot directly use the 2D image structure in its first Dense layer.

It therefore **flattens** the 28 × 28 image:

`28 × 28 → 784`

After flattening, the model receives one long vector of numbers.


In [ ]:
image = X_train[0]
flat_image = image.flatten()

print("Original image shape:", image.shape)
print("Flattened DNN input:", flat_image.shape)

plt.figure(figsize=(12, 3))

plt.subplot(1, 2, 1)
plt.imshow(image, cmap="gray")
plt.title("Original image — 28 × 28")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(flat_image.reshape(1, -1), cmap="gray", aspect="auto")
plt.title("After Flatten — 1 × 784")
plt.yticks([])

plt.show()


## 6. Build the fully connected DNN

The DNN pipeline is:

`Image → Flatten → Dense → Dense → Softmax`

It is a straightforward fully connected neural network.


In [ ]:
dnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
], name="DNN")

dnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

dnn.summary()


## 7. Build the CNN

The CNN keeps the 2D image structure while it extracts features.

Pipeline:

`Image → Conv2D → MaxPooling → Flatten → Dense → Softmax`

We use the **Functional API** here because it also lets us later expose the convolution layer and visualize its feature maps.


In [ ]:
# Functional API
inputs = tf.keras.Input(shape=(28, 28, 1), name="image")

conv_layer = tf.keras.layers.Conv2D(
    filters=32,
    kernel_size=(3, 3),
    activation="relu",
    name="conv2d"
)

x = conv_layer(inputs)
x = tf.keras.layers.MaxPooling2D((2, 2), name="max_pool")(x)
x = tf.keras.layers.Flatten(name="flatten")(x)
x = tf.keras.layers.Dense(128, activation="relu", name="dense_128")(x)
outputs = tf.keras.layers.Dense(10, activation="softmax", name="output")(x)

cnn = tf.keras.Model(inputs=inputs, outputs=outputs, name="CNN")

cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.summary()


## 8. Compare the architectures and parameters

This is one of the most important observations.

For the DNN, the first Dense layer receives all **784 pixels**.

For the CNN, the first convolution uses small **3 × 3 filters** shared across the image.

This demonstrates two core CNN ideas:

- **Local connectivity** — a filter looks at a small region.
- **Parameter sharing** — the same filter is reused across different image locations.


In [ ]:
print(f"DNN trainable parameters: {dnn.count_params():,}")
print(f"CNN trainable parameters: {cnn.count_params():,}")

dnn_first_dense_params = dnn.layers[1].count_params()
cnn_conv_params = conv_layer.count_params()

print()
print(f"DNN first Dense layer parameters: {dnn_first_dense_params:,}")
print(f"CNN Conv2D layer parameters:      {cnn_conv_params:,}")


## 9. Train both models on the same data

The training pipeline is the same:

`compile → fit → evaluate`

Only the model architecture and corresponding input shape differ.

We use three epochs to keep this a learning experiment rather than an accuracy competition.


In [ ]:
history_dnn = dnn.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

history_cnn = cnn.fit(
    X_train_cnn,
    y_train,
    epochs=3,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)


## 10. Compare test performance

The models are evaluated on the **same test set**.

Accuracy is useful here, but remember: the main learning objective is understanding **why the architectures behave differently**, not chasing a particular score.


In [ ]:
dnn_test_loss, dnn_test_acc = dnn.evaluate(
    X_test, y_test, verbose=0
)

cnn_test_loss, cnn_test_acc = cnn.evaluate(
    X_test_cnn, y_test, verbose=0
)

print(f"DNN test accuracy: {dnn_test_acc:.4f}")
print(f"CNN test accuracy: {cnn_test_acc:.4f}")


## 11. Compare predictions on the exact same images

Both models receive the same test images.

The DNN receives the flattened representation internally.

The CNN receives the 2D image and extracts spatial features before flattening.


In [ ]:
def predict_one(index):
    dnn_probs = dnn.predict(X_test[index:index+1], verbose=0)
    cnn_probs = cnn.predict(X_test_cnn[index:index+1], verbose=0)

    dnn_pred = np.argmax(dnn_probs, axis=1)[0]
    cnn_pred = np.argmax(cnn_probs, axis=1)[0]

    plt.figure(figsize=(4, 4))
    plt.imshow(X_test[index], cmap="gray")
    plt.title(
        f"Actual: {y_test[index]} | "
        f"DNN: {dnn_pred} | CNN: {cnn_pred}"
    )
    plt.axis("off")
    plt.show()

    print("DNN probabilities:", np.round(dnn_probs[0], 3))
    print("CNN probabilities:", np.round(cnn_probs[0], 3))

# Try different test images by changing the index.
predict_one(0)


## 12. Visualize CNN feature maps

This is the key visualization.

The first convolution layer has **32 filters**.

Each filter learns a different useful pattern during training. The resulting feature maps show where those learned patterns activate in the image.

This is what makes the CNN fundamentally different from simply flattening the image at the start.


In [ ]:
# Create a model that exposes the trained Conv2D output.
# Because the CNN was built with the Functional API, this is straightforward.
feature_model = tf.keras.Model(
    inputs=cnn.input,
    outputs=conv_layer.output,
    name="CNN_Feature_Extractor"
)

sample = X_test_cnn[0:1]

feature_maps = feature_model.predict(sample, verbose=0)

print("Input shape:", sample.shape)
print("Feature map shape:", feature_maps.shape)


In [ ]:
# Visualize the first 16 of the 32 feature maps
plt.figure(figsize=(10, 10))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(feature_maps[0, :, :, i], cmap="gray")
    plt.title(f"Feature map {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## 13. What the feature maps mean

Do not interpret every feature map as a simple human-readable object such as "this is the vertical-edge filter." The filters are **learned automatically** and may respond to combinations of shapes or patterns.

The important progression is:

`Pixels → local patterns → feature maps → higher-level features → classification`

This is the core intuition behind CNNs.


# Final Takeaway

### Fully Connected DNN

```text
28 × 28 image
     ↓
Flatten
     ↓
784 values
     ↓
Dense
     ↓
Dense
     ↓
Prediction
```

### CNN

```text
28 × 28 × 1 image
       ↓
   Conv2D
       ↓
 Feature maps
       ↓
 MaxPooling
       ↓
   Flatten
       ↓
    Dense
       ↓
  Prediction
```

### What stays the same?

- Same dataset
- Same preprocessing idea
- Same train/test split
- Same compile/train/evaluate workflow
- Same classification objective

### What changes?

The **model architecture**.

The DNN treats the image as a flattened vector for its first Dense layer.

The CNN first exploits the image's **spatial structure** using:

- local receptive fields
- shared filters
- convolution
- pooling

### Interview-ready statement

> **A fully connected DNN typically flattens an image and processes all pixels through Dense layers. A CNN preserves spatial structure and uses local, shared convolutional filters to learn spatial features efficiently before classification.**
